# 🧠 MBTI Personality Detection — Baselines

---
**Datasets supported:**
- MBTI-500 (~106K rows): `/kaggle/input/.../MBTI 500.csv`
- Kaggle MBTI (~8600 rows): `/kaggle/input/.../mbti_1.csv`

## 📦 Cell 1 — Install Dependencies

In [ ]:
!pip install -q transformers faiss-cpu scikit-learn torch torchvision torchaudio
!pip install -q torch_geometric sentence-transformers accelerate tqdm openai

## 📥 Cell 2 — Imports

In [ ]:
import os, re, json, time, warnings, gc
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from transformers import AutoTokenizer, AutoModel
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.multioutput import MultiOutputClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, classification_report
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_class_weight

import faiss

try:
    from torch_geometric.nn import GCNConv, global_mean_pool
    from torch_geometric.data import Data, Batch
    HAS_PYG = True
except ImportError:
    HAS_PYG = False
    print('⚠️  torch_geometric not available — D-DGCN will be skipped')

warnings.filterwarnings('ignore')
print('✅ Imports done')

## ⚙️ Cell 3 — Config

In [ ]:
# ───────────────────────────────────────────────
# CONFIG — all paths and hyperparameters here
# ───────────────────────────────────────────────
CONFIG = {
    # Paths
    'DATA_PATH_500':  '/kaggle/input/datasets/zeyadkhalid/mbti-personality-types-500-dataset/MBTI 500.csv',
    'DATA_PATH_8K':   '/kaggle/input/datasets/datasnaek/mbti-type/mbti_1.csv',
    'RESULT_DIR':     '/kaggle/working/results',

    # Data preprocessing
    'MAX_POSTS':      50,
    'MAX_WORDS':      70,

    # Model
    'ROBERTA_NAME':   'roberta-base',
    'MAX_TOK_LEN':    128,
    'BATCH_SIZE':     16,           # ⚡ T4×2: tăng từ 8 → 16 (gradient checkpointing cho phép)
    'EVAL_BATCH_SIZE': 32,          # ⚡ Eval không cần gradient → batch lớn hơn 2x
    'ENCODER_LR':     1e-5,
    'OTHER_LR':       1e-3,
    'MAX_EPOCHS':     10,
    'PATIENCE':       3,

    # GCN
    'GCN_HIDDEN':     128,
    'GCN_OUT':        64,
    'TOP_K_GRAPH':    10,

    # RAG
    'RAG_TOP_K':      5,
    'EMBED_DIM':      768,

    # LLM
    'LLM_PROVIDER':   'openai',   # 'openai' | 'deepseek' | 'local'
    'LLM_MODEL':      'gpt-4o-mini',
    'OPENAI_API_KEY': os.environ.get('OPENAI_API_KEY', ''),
    'DEEPSEEK_KEY':   os.environ.get('DEEPSEEK_API_KEY', ''),
}

os.makedirs(CONFIG['RESULT_DIR'], exist_ok=True)

# ── GPU detection + CUDA optimizations ──
NUM_GPUS = torch.cuda.device_count()
DEVICE   = torch.device('cuda' if NUM_GPUS > 0 else 'cpu')

if DEVICE.type == 'cuda':
    # ⚡ cuDNN auto-tuner: tìm convolution algorithm nhanh nhất cho input size cố định
    torch.backends.cudnn.benchmark = True
    # ⚡ TF32: tăng tốc matmul trên Ampere+ (T4 = Turing, không có TF32 nhưng không hại)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print(f'🖥  GPUs available: {NUM_GPUS}  |  Device: {DEVICE}')
if NUM_GPUS == 2:
    print('🚀 Kaggle T4×2 detected → DataParallel + batch_size={}'.format(CONFIG['BATCH_SIZE']))
elif NUM_GPUS == 1:
    print('🚀 Single GPU detected → standard training')
else:
    print('⚠️  No GPU found → CPU mode')

# ── Constants ──
MBTI_TYPES  = ['infj','infp','intj','intp','isfj','isfp','istj','istp',
               'enfj','enfp','entj','entp','esfj','esfp','estj','estp']
MBTI_EXTRA  = ['introvert','extrovert','introverted','extroverted',
               'sensing','intuition','thinking','feeling','judging','perceiving']
ALL_MASK    = MBTI_TYPES + MBTI_EXTRA
MASK_RE     = re.compile(r'\b(' + '|'.join(ALL_MASK) + r')\b', re.IGNORECASE)

LABEL_COL   = 'type'
TEXT_COL    = 'posts'
LABEL_COLS  = ['label_ie', 'label_sn', 'label_tf', 'label_jp']
DIM_NAMES   = ['I/E', 'S/N', 'T/F', 'J/P']
print('✅ Config ready')

## 📂 Cell 4 — Load & Preprocess Dataset

In [ ]:
# ── Load dataset (prefer 500K, fall back to 8K) ──
def load_dataset(cfg):
    for path in [cfg['DATA_PATH_500'], cfg['DATA_PATH_8K']]:
        try:
            df = pd.read_csv(path)
            print(f'✅ Loaded: {path}  shape={df.shape}')
            return df
        except FileNotFoundError:
            continue
    raise FileNotFoundError('❌ No dataset found. Add data and check paths.')

df_raw = load_dataset(CONFIG)


# ── Step 1: Map 16-class MBTI → 4 binary labels ──
def map_mbti_to_binary(mbti_str):
    m = mbti_str.strip().upper()
    return int(m[0]=='I'), int(m[1]=='N'), int(m[2]=='F'), int(m[3]=='J')

df_raw[['label_ie','label_sn','label_tf','label_jp']] = (
    df_raw[LABEL_COL].apply(lambda x: pd.Series(map_mbti_to_binary(x)))
)


# ── Step 2: Psycholinguistic masking + truncation ──
def preprocess_user_posts(raw_str, max_posts=CONFIG['MAX_POSTS'], max_words=CONFIG['MAX_WORDS']):
    """Split posts, mask MBTI keywords, truncate to max_words each."""
    posts = raw_str.split('|||')
    result = []
    for p in posts[:max_posts]:
        p = MASK_RE.sub('<type>', p.strip())
        p = ' '.join(p.split()[:max_words])
        if p:
            result.append(p)
    return result

print('⏳ Preprocessing posts (masking + truncation)...')
df_raw['processed_posts'] = df_raw[TEXT_COL].apply(preprocess_user_posts)

# Concatenated version for SVM / RAG text input
df_raw['concat_posts'] = df_raw['processed_posts'].apply(lambda x: ' ||| '.join(x))
print(f'✅ Preprocessing done. Rows: {len(df_raw)}')


# ── Step 3: Stratified split across ALL 4 axes jointly ──
df_raw['combo_label'] = df_raw[LABEL_COLS].astype(str).agg(''.join, axis=1)

train_df, temp_df = train_test_split(
    df_raw, test_size=0.4, stratify=df_raw['combo_label'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['combo_label'], random_state=42
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f'\n📊 Split → Train:{len(train_df)} | Val:{len(val_df)} | Test:{len(test_df)}')


# ── Step 4: Class weights from train set ──
class_weights_list = []
for col in LABEL_COLS:
    y = train_df[col].values
    cw = compute_class_weight('balanced', classes=np.unique(y), y=y)
    class_weights_list.append(torch.tensor(cw, dtype=torch.float))
print('✅ Class weights computed:', [cw.tolist() for cw in class_weights_list])

## 📈 Cell 12 — Baseline 1: SVM + TF-IDF (LinearSVC — tối ưu tốc độ)

**Tối ưu so với SVC(probability=True):**
- `LinearSVC` dùng **liblinear** (nhanh hơn 5-10x so với libsvm cho linear kernel trên sparse data)
- `CalibratedClassifierCV(cv=3)`: Platt scaling chỉ 3-fold (thay vì 5-fold mặc định của SVC)
- `max_iter=2000`: đủ cho convergence, tránh warning

In [ ]:
print('⏳ [SVM] Training (LinearSVC + CalibratedClassifierCV)...')
t0_svm = time.time()

vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), sublinear_tf=True)

# ⚡ LinearSVC: dùng liblinear, nhanh hơn 5-10x so với SVC(kernel='linear') trên sparse data
# ⚡ CalibratedClassifierCV(cv=3): chỉ 3-fold Platt scaling (vs 5-fold mặc định SVC)
base_svm = LinearSVC(class_weight='balanced', C=1.0, max_iter=2000)
svm_model = MultiOutputClassifier(
    CalibratedClassifierCV(base_svm, cv=3, method='sigmoid'),
    n_jobs=-1
)

X_train_text = train_df['concat_posts'].tolist()
X_test_text  = test_df['concat_posts'].tolist()
y_train_svm  = train_df[LABEL_COLS].values
y_test_svm   = test_df[LABEL_COLS].values

# 1. Fit và Transform
X_train_feat = vectorizer.fit_transform(X_train_text)
X_test_feat  = vectorizer.transform(X_test_text)

# 2. Fix sparse matrix read-only error
X_train_feat.sort_indices()
X_test_feat.sort_indices()

# 3. Fit mô hình
svm_model.fit(X_train_feat, y_train_svm)
svm_preds = svm_model.predict(X_test_feat)

# 4. Extract probabilities (probability of class 1 for each axis)
svm_proba_list = svm_model.predict_proba(X_test_feat)
svm_probs = np.column_stack([p[:, 1] for p in svm_proba_list])  # [n_test, 4]

t_svm = time.time() - t0_svm

# 5. Reconstruct 16-type predictions
def preds_to_type(ie, sn, tf, jp):
    return ('I' if ie else 'E') + ('N' if sn else 'S') + ('F' if tf else 'T') + ('J' if jp else 'P')

svm_pred_types = [preds_to_type(*row) for row in svm_preds]
svm_true_types = test_df[LABEL_COL].str.upper().tolist()

# ── Save SVM predictions ──
svm_save = pd.DataFrame()
for i, c in enumerate(LABEL_COLS):
    svm_save[f'y_true_{c}'] = y_test_svm[:, i]
    svm_save[f'y_pred_{c}'] = svm_preds[:, i]
    svm_save[f'y_prob_{c}'] = np.round(svm_probs[:, i], 4)
svm_save['y_true_type'] = svm_true_types
svm_save['y_pred_type'] = svm_pred_types
svm_save.to_csv(f"{CONFIG['RESULT_DIR']}/svm_predictions.csv", index=False)

svm_scores = {}
for i, name in enumerate(DIM_NAMES):
    svm_scores[name] = f1_score(y_test_svm[:, i], svm_preds[:, i], average='macro', zero_division=0)
svm_scores['Average'] = np.mean(list(svm_scores.values()))
print(f'✅ SVM done in {t_svm:.1f}s — scores: {svm_scores}')
print(f'   Saved → {CONFIG["RESULT_DIR"]}/svm_predictions.csv')

## 🗂️ Cell 13 — Dataset & DataLoader (Pre-tokenized + Optimized I/O)

**Tối ưu so với bản gốc:**
- **Pre-tokenize toàn bộ dataset** 1 lần thay vì tokenize trong `__getitem__` mỗi epoch → tiết kiệm ~60% DataLoader time
- `num_workers=4` (tăng từ 2) + `persistent_workers=True` → worker processes không bị tạo lại mỗi epoch
- `prefetch_factor=2`: prefetch 2 batch tiếp theo trong khi GPU đang xử lý
- `EVAL_BATCH_SIZE=32` cho eval/test (không cần gradient → dùng batch lớn hơn)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG['ROBERTA_NAME'])


class MBTIDataset(Dataset):
    """
    ⚡ Pre-tokenized dataset — tokenize 1 lần tại __init__, không tokenize lại mỗi __getitem__.
    Tradeoff: dùng thêm RAM (~2-3GB cho 100K samples) nhưng tiết kiệm ~60% DataLoader time.
    """
    def __init__(self, dataframe, tok, max_posts=CONFIG['MAX_POSTS'],
                 max_length=CONFIG['MAX_TOK_LEN']):
        self.max_posts  = max_posts
        self.max_length = max_length

        df = dataframe.reset_index(drop=True)
        self.labels = torch.tensor(df[LABEL_COLS].values, dtype=torch.float)

        # ⚡ Pre-tokenize tất cả samples một lần
        print(f'  ⏳ Pre-tokenizing {len(df)} samples...')
        self.all_ids   = []
        self.all_masks = []
        for idx in tqdm(range(len(df)), desc='  Tokenize', leave=False):
            posts = df.iloc[idx]['processed_posts'][:max_posts]
            enc = tok(
                posts, padding='max_length', truncation=True,
                max_length=max_length, return_tensors='pt'
            )
            ids   = enc['input_ids']       # [P, L]
            masks = enc['attention_mask']   # [P, L]

            P = ids.shape[0]
            if P < max_posts:
                pad_ids   = torch.zeros(max_posts - P, max_length, dtype=torch.long)
                pad_masks = torch.zeros(max_posts - P, max_length, dtype=torch.long)
                ids   = torch.cat([ids,   pad_ids],   dim=0)
                masks = torch.cat([masks, pad_masks], dim=0)

            self.all_ids.append(ids)
            self.all_masks.append(masks)
        print(f'  ✅ Pre-tokenized {len(df)} samples')

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.all_ids[idx],
            'attention_mask': self.all_masks[idx],
            'labels':         self.labels[idx],
        }


print('⏳ Building pre-tokenized datasets...')
train_ds = MBTIDataset(train_df, tokenizer)
val_ds   = MBTIDataset(val_df,   tokenizer)
test_ds  = MBTIDataset(test_df,  tokenizer)

# ⚡ DataLoader optimizations: num_workers=4, persistent_workers, prefetch
NW = 4
train_loader = DataLoader(
    train_ds, batch_size=CONFIG['BATCH_SIZE'], shuffle=True,
    num_workers=NW, pin_memory=True, persistent_workers=True, prefetch_factor=2
)
val_loader = DataLoader(
    val_ds, batch_size=CONFIG['EVAL_BATCH_SIZE'], shuffle=False,
    num_workers=NW, pin_memory=True, persistent_workers=True, prefetch_factor=2
)
test_loader = DataLoader(
    test_ds, batch_size=CONFIG['EVAL_BATCH_SIZE'], shuffle=False,
    num_workers=NW, pin_memory=True, persistent_workers=True, prefetch_factor=2
)

print(f'✅ DataLoaders ready — train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}')
print(f'   Train batch_size={CONFIG["BATCH_SIZE"]} | Eval batch_size={CONFIG["EVAL_BATCH_SIZE"]}')
print(f'   num_workers={NW} | persistent_workers=True | prefetch_factor=2')

## 🤗 Cell 14 — Baseline 2: RoBERTa-mean (T4×2 Optimized)

**Tối ưu T4×2:**
- `torch.compile()` (PyTorch 2.0+): **JIT compile** model → fuse operators, giảm kernel launch overhead → ~20-30% speedup
- `non_blocking=True` trên `.to(DEVICE)`: **overlap** CPU→GPU transfer với GPU computation
- `torch.cuda.amp.autocast`: **FP16 mixed precision** — T4 có Tensor Cores cho FP16, ~2x throughput
- `DataParallel`: **split batch** qua 2 GPU tự động
- `EVAL_BATCH_SIZE=32`: eval không cần gradient → VRAM đủ cho batch lớn hơn

In [ ]:
class RoBERTaMeanModel(nn.Module):
    """
    Tensor flow:
    [B,P=50,L] → reshape [B*P,L] → RoBERTa CLS → [B*P,768]
    → reshape [B,P,768] → mean → [B,768] → 4 heads → [B,4]
    """
    def __init__(self, model_name=CONFIG['ROBERTA_NAME'], num_dims=4, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.encoder.gradient_checkpointing_enable()
        h = self.encoder.config.hidden_size
        self.drop  = nn.Dropout(dropout)
        self.heads = nn.ModuleList([nn.Linear(h, 1) for _ in range(num_dims)])

    def forward(self, input_ids, attention_mask, **kwargs):
        B, P, L = input_ids.shape
        ids   = input_ids.view(B*P, L)
        masks = attention_mask.view(B*P, L)
        out   = self.encoder(input_ids=ids, attention_mask=masks)
        cls   = out.last_hidden_state[:, 0, :]     # [B*P, 768]
        user  = cls.view(B, P, -1).mean(dim=1)    # [B, 768]
        user  = self.drop(user)
        return torch.cat([h(user) for h in self.heads], dim=-1)  # [B, 4]


class NeuralTrainer:
    """⚡ Optimized trainer: torch.compile, non_blocking transfers, fp16, DataParallel."""

    def __init__(self, model, train_loader, val_loader, class_weights,
                 encoder_lr=1e-5, other_lr=1e-3, patience=3, use_fp16=True,
                 compile_model=True):
        self.raw_model = model
        if NUM_GPUS > 1:
            model = nn.DataParallel(model)
            print(f'  🔀 DataParallel on {NUM_GPUS} GPUs')
        self.model = model.to(DEVICE)

        # ⚡ torch.compile: JIT fuse operators → ~20-30% speedup
        if compile_model and hasattr(torch, 'compile'):
            try:
                self.model = torch.compile(self.model, mode='reduce-overhead')
                print(f'  ⚡ torch.compile enabled (reduce-overhead mode)')
            except Exception as e:
                print(f'  ⚠️ torch.compile failed: {e} — using eager mode')

        self.train_loader = train_loader
        self.val_loader   = val_loader
        self.patience     = patience
        self.scaler       = GradScaler() if use_fp16 and DEVICE.type == 'cuda' else None
        self.use_fp16     = use_fp16 and DEVICE.type == 'cuda'

        # Differential LR
        enc_params = list(self.raw_model.encoder.parameters())
        enc_ids    = set(id(p) for p in enc_params)
        other_p    = [p for p in self.raw_model.parameters() if id(p) not in enc_ids]
        self.opt   = torch.optim.AdamW([
            {'params': enc_params, 'lr': encoder_lr},
            {'params': other_p,    'lr': other_lr},
        ], weight_decay=1e-2)

        # Per-axis BCEWithLogitsLoss with pos_weight
        self.criteria = []
        for cw in class_weights:
            pw = (cw[1]/cw[0]).to(DEVICE) if len(cw) == 2 else torch.tensor(1.0).to(DEVICE)
            self.criteria.append(nn.BCEWithLogitsLoss(pos_weight=pw))

    def _loss(self, logits, labels):
        return sum(c(logits[:, i], labels[:, i])
                   for i, c in enumerate(self.criteria)) / len(self.criteria)

    def train_epoch(self):
        self.model.train()
        total = 0
        for batch in tqdm(self.train_loader, leave=False, desc='Train'):
            # ⚡ non_blocking=True: overlap CPU→GPU transfer với computation
            ids   = batch['input_ids'].to(DEVICE, non_blocking=True)
            masks = batch['attention_mask'].to(DEVICE, non_blocking=True)
            lbls  = batch['labels'].to(DEVICE, non_blocking=True)
            self.opt.zero_grad(set_to_none=True)  # ⚡ set_to_none=True: tránh memset, nhẹ hơn zero_grad()
            if self.use_fp16:
                with autocast():
                    logits = self.model(ids, masks)
                    loss   = self._loss(logits, lbls)
                self.scaler.scale(loss).backward()
                self.scaler.unscale_(self.opt)
                nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                self.scaler.step(self.opt)
                self.scaler.update()
            else:
                logits = self.model(ids, masks)
                loss   = self._loss(logits, lbls)
                loss.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                self.opt.step()
            total += loss.item()
        return total / len(self.train_loader)

    @torch.no_grad()
    def evaluate(self, loader):
        self.model.eval()
        all_logits, all_lbls = [], []
        for batch in tqdm(loader, leave=False, desc='Eval'):
            ids   = batch['input_ids'].to(DEVICE, non_blocking=True)
            masks = batch['attention_mask'].to(DEVICE, non_blocking=True)
            if self.use_fp16:
                with autocast():
                    logits = self.model(ids, masks)
            else:
                logits = self.model(ids, masks)
            all_logits.append(logits.cpu().float())
            all_lbls.append(batch['labels'])
        logits_all = torch.cat(all_logits)
        lbls_all   = torch.cat(all_lbls).int().numpy()
        preds_all  = (torch.sigmoid(logits_all) >= 0.5).int().numpy()
        probs_all  = torch.sigmoid(logits_all).numpy()
        scores = {n: f1_score(lbls_all[:, i], preds_all[:, i], average='macro', zero_division=0)
                  for i, n in enumerate(DIM_NAMES)}
        scores['Average'] = np.mean(list(scores.values()))
        return lbls_all, preds_all, probs_all, scores

    def train(self, max_epochs=CONFIG['MAX_EPOCHS']):
        best_f1, patience_cnt, best_state = 0, 0, None
        for ep in range(1, max_epochs+1):
            t_ep = time.time()
            loss = self.train_epoch()
            _, _, _, val_sc = self.evaluate(self.val_loader)
            dt = time.time() - t_ep
            print(f'  Ep{ep:02d} loss={loss:.4f} val_avg={val_sc["Average"]:.4f} ({dt:.1f}s)')
            if val_sc['Average'] > best_f1:
                best_f1, patience_cnt = val_sc['Average'], 0
                best_state = {k: v.clone() for k, v in self.raw_model.state_dict().items()}
            else:
                patience_cnt += 1
                if patience_cnt >= self.patience:
                    print(f'  ⏹ Early stop at ep{ep} (best={best_f1:.4f})')
                    break
            torch.cuda.empty_cache()
        if best_state:
            self.raw_model.load_state_dict(best_state)


# ── Train RoBERTa ──
print('\n⏳ [RoBERTa-mean] Training...')
t0_rob = time.time()
roberta_model   = RoBERTaMeanModel()
roberta_trainer = NeuralTrainer(
    roberta_model, train_loader, val_loader, class_weights_list,
    encoder_lr=CONFIG['ENCODER_LR'], other_lr=CONFIG['OTHER_LR'],
    patience=CONFIG['PATIENCE']
)
roberta_trainer.train()
t_rob_train = time.time() - t0_rob

rob_true, rob_preds, rob_probs, rob_scores = roberta_trainer.evaluate(test_loader)

# ── Reconstruct 16-type predictions ──
def preds_to_type(ie, sn, tf, jp):
    return ('I' if ie else 'E') + ('N' if sn else 'S') + ('F' if tf else 'T') + ('J' if jp else 'P')

rob_pred_types = [preds_to_type(*row) for row in rob_preds]
rob_true_types = test_df[LABEL_COL].str.upper().tolist()

# ── Save RoBERTa predictions ──
rob_save = pd.DataFrame()
for i, c in enumerate(LABEL_COLS):
    rob_save[f'y_true_{c}'] = rob_true[:, i]
    rob_save[f'y_pred_{c}'] = rob_preds[:, i]
    rob_save[f'y_prob_{c}'] = np.round(rob_probs[:, i], 4)
rob_save['y_true_type'] = rob_true_types
rob_save['y_pred_type'] = rob_pred_types
rob_save.to_csv(f"{CONFIG['RESULT_DIR']}/roberta_predictions.csv", index=False)
print(f'✅ RoBERTa done in {t_rob_train:.1f}s — scores: {rob_scores}')
print(f'   Saved → {CONFIG["RESULT_DIR"]}/roberta_predictions.csv')

# ⚡ Free RoBERTa memory before D-DGCN
del roberta_model, roberta_trainer
torch.cuda.empty_cache()
gc.collect()

## 🕸️ Cell 15 — Baseline 3: D-DGCN (T4×2 Optimized)

**Tối ưu tương tự RoBERTa**: `torch.compile`, `non_blocking`, `DataParallel`, FP16, `EVAL_BATCH_SIZE=32`

In [ ]:
if not HAS_PYG:
    print('⚠️  Skipping D-DGCN (torch_geometric not available)')
    ddgcn_scores = {n: 0.0 for n in DIM_NAMES + ['Average']}
    ddgcn_true   = np.zeros((len(test_df), 4))
    ddgcn_preds  = np.zeros((len(test_df), 4))
    ddgcn_probs  = np.full((len(test_df), 4), 0.5)
    t_dgcn_train = 0.0
else:
    class DDGCNModel(nn.Module):
        """
        Tensor flow:
        [B,P=50,L]
          → RoBERTa CLS : [B,P,768]
          → Project     : [B,P,256]
          → Per-sample cosine-sim top-K graph → PyG Batch
          → GCNConv(256→128) → ReLU
          → GCNConv(128→64)  → ReLU
          → global_mean_pool : [B,64]
          → 4 heads          : [B,4]
        """
        def __init__(self, model_name=CONFIG['ROBERTA_NAME'],
                     proj_dim=256, gcn_h=CONFIG['GCN_HIDDEN'],
                     gcn_out=CONFIG['GCN_OUT'], top_k=CONFIG['TOP_K_GRAPH'],
                     dropout=0.1):
            super().__init__()
            self.top_k   = top_k
            self.encoder = AutoModel.from_pretrained(model_name)
            self.encoder.gradient_checkpointing_enable()
            roberta_h = self.encoder.config.hidden_size
            self.proj = nn.Linear(roberta_h, proj_dim)
            self.drop = nn.Dropout(dropout)
            self.gcn1 = GCNConv(proj_dim, gcn_h)
            self.gcn2 = GCNConv(gcn_h,    gcn_out)
            self.heads = nn.ModuleList([nn.Linear(gcn_out, 1) for _ in range(4)])

        def _build_graph(self, feats):
            """feats: [P, D] → edge_index: [2, edges]"""
            norm  = F.normalize(feats, dim=-1)
            sim   = torch.mm(norm, norm.t())           # [P, P]
            P     = sim.shape[0]
            k     = min(self.top_k, P-1)
            sim.fill_diagonal_(torch.finfo(sim.dtype).min)
            _, topk_idx = sim.topk(k, dim=-1)          # [P, k]
            src = torch.arange(P, device=feats.device).unsqueeze(1).expand(-1, k).reshape(-1)
            dst = topk_idx.reshape(-1)
            edge_index = torch.stack([
                torch.cat([src, dst]),
                torch.cat([dst, src])
            ], dim=0)
            return edge_index

        def forward(self, input_ids, attention_mask, **kwargs):
            B, P, L = input_ids.shape
            ids   = input_ids.view(B*P, L)
            masks = attention_mask.view(B*P, L)
            out   = self.encoder(input_ids=ids, attention_mask=masks)
            cls   = out.last_hidden_state[:, 0, :]      # [B*P, 768]
            feats = self.drop(F.relu(self.proj(cls)))   # [B*P, 256]
            feats = feats.view(B, P, -1)                # [B, P, 256]

            data_list = []
            for b in range(B):
                ei = self._build_graph(feats[b])
                data_list.append(Data(x=feats[b], edge_index=ei))
            pg = Batch.from_data_list(data_list)

            x  = F.relu(self.gcn1(pg.x, pg.edge_index))  # [B*P, 128]
            x  = self.drop(x)
            x  = F.relu(self.gcn2(x,    pg.edge_index))  # [B*P, 64]
            ue = global_mean_pool(x, pg.batch)            # [B, 64]
            ue = self.drop(ue)
            return torch.cat([h(ue) for h in self.heads], dim=-1)  # [B, 4]

    print('\n⏳ [D-DGCN] Training...')
    t0_dgcn = time.time()
    ddgcn_model   = DDGCNModel()
    # ⚡ D-DGCN: compile_model=False vì dynamic graph construction không tương thích tốt với torch.compile
    ddgcn_trainer = NeuralTrainer(
        ddgcn_model, train_loader, val_loader, class_weights_list,
        encoder_lr=CONFIG['ENCODER_LR'], other_lr=CONFIG['OTHER_LR'],
        patience=CONFIG['PATIENCE'], compile_model=False
    )
    ddgcn_trainer.train()
    t_dgcn_train = time.time() - t0_dgcn

    ddgcn_true, ddgcn_preds, ddgcn_probs, ddgcn_scores = ddgcn_trainer.evaluate(test_loader)

    # ── Reconstruct 16-type predictions ──
    ddgcn_pred_types = [preds_to_type(*row) for row in ddgcn_preds]
    ddgcn_true_types = test_df[LABEL_COL].str.upper().tolist()

    # ── Save D-DGCN predictions ──
    dgcn_save = pd.DataFrame()
    for i, c in enumerate(LABEL_COLS):
        dgcn_save[f'y_true_{c}'] = ddgcn_true[:, i]
        dgcn_save[f'y_pred_{c}'] = ddgcn_preds[:, i]
        dgcn_save[f'y_prob_{c}'] = np.round(ddgcn_probs[:, i], 4)
    dgcn_save['y_true_type'] = ddgcn_true_types
    dgcn_save['y_pred_type'] = ddgcn_pred_types
    dgcn_save.to_csv(f"{CONFIG['RESULT_DIR']}/ddgcn_predictions.csv", index=False)
    print(f'✅ D-DGCN done in {t_dgcn_train:.1f}s — scores: {ddgcn_scores}')
    print(f'   Saved → {CONFIG["RESULT_DIR"]}/ddgcn_predictions.csv')

    # ⚡ Free D-DGCN memory
    del ddgcn_model, ddgcn_trainer
    torch.cuda.empty_cache()
    gc.collect()

## 📊 Cell 16 — Final Comparison Table (Macro F1, Accuracy, AUC-ROC, 16-Type Metrics)

In [ ]:
def build_full_comparison(result_dir=CONFIG['RESULT_DIR']):
    """
    Read all prediction CSVs and compute comprehensive metrics:
      - Per-axis: Macro F1, Accuracy, AUC-ROC
      - 16-type:  Accuracy, Macro F1, Weighted F1, AUC-ROC (Macro OvR)
    """
    files = {
        'SVM + TF-IDF': 'svm_predictions.csv',
        'RoBERTa-mean': 'roberta_predictions.csv',
        'D-DGCN':       'ddgcn_predictions.csv',
    }

    DIM_LABEL_MAP = {
        'ie': {'I': 1, 'E': 0},
        'sn': {'N': 1, 'S': 0},
        'tf': {'F': 1, 'T': 0},
        'jp': {'J': 1, 'P': 0},
    }

    all_rows_axis = []
    all_rows_16   = []

    for name, fname in files.items():
        fpath = os.path.join(result_dir, fname)
        if not os.path.exists(fpath):
            print(f'  ⚠️  {fname} not found, skipping.')
            continue
        df_r = pd.read_csv(fpath)

        has_probs = f'y_prob_{LABEL_COLS[0]}' in df_r.columns
        has_types = 'y_true_type' in df_r.columns

        # ═══════════════ PER-AXIS METRICS ═══════════════
        row_axis = {'Model': name}
        f1s, accs, aucs = [], [], []
        for col, dim in zip(LABEL_COLS, DIM_NAMES):
            y_true = df_r[f'y_true_{col}'].values
            y_pred = df_r[f'y_pred_{col}'].values

            f1  = f1_score(y_true, y_pred, average='macro', zero_division=0)
            acc = accuracy_score(y_true, y_pred)
            f1s.append(f1)
            accs.append(acc)

            row_axis[f'F1 {dim}']  = round(f1, 4)
            row_axis[f'Acc {dim}'] = round(acc, 4)

            if has_probs:
                y_prob = df_r[f'y_prob_{col}'].values
                try:
                    auc = roc_auc_score(y_true, y_prob)
                except ValueError:
                    auc = 0.5
                row_axis[f'AUC {dim}'] = round(auc, 4)
                aucs.append(auc)

        row_axis['Avg F1']  = round(np.mean(f1s), 4)
        row_axis['Avg Acc'] = round(np.mean(accs), 4)
        if aucs:
            row_axis['Avg AUC'] = round(np.mean(aucs), 4)

        all_rows_axis.append(row_axis)

        # ═══════════════ 16-TYPE METRICS ═══════════════
        if has_types:
            y_true_type = df_r['y_true_type'].str.upper().values
            y_pred_type = df_r['y_pred_type'].str.upper().values
        else:
            y_true_type = np.array([
                preds_to_type(
                    df_r[f'y_true_{LABEL_COLS[0]}'].iloc[i],
                    df_r[f'y_true_{LABEL_COLS[1]}'].iloc[i],
                    df_r[f'y_true_{LABEL_COLS[2]}'].iloc[i],
                    df_r[f'y_true_{LABEL_COLS[3]}'].iloc[i],
                ) for i in range(len(df_r))
            ])
            y_pred_type = np.array([
                preds_to_type(
                    df_r[f'y_pred_{LABEL_COLS[0]}'].iloc[i],
                    df_r[f'y_pred_{LABEL_COLS[1]}'].iloc[i],
                    df_r[f'y_pred_{LABEL_COLS[2]}'].iloc[i],
                    df_r[f'y_pred_{LABEL_COLS[3]}'].iloc[i],
                ) for i in range(len(df_r))
            ])

        type_to_idx = {t.upper(): i for i, t in enumerate(MBTI_TYPES)}
        y_true_16 = np.array([type_to_idx.get(t, 0) for t in y_true_type])
        y_pred_16 = np.array([type_to_idx.get(t, 0) for t in y_pred_type])

        f1_16_macro    = f1_score(y_true_16, y_pred_16, average='macro', zero_division=0)
        f1_16_weighted = f1_score(y_true_16, y_pred_16, average='weighted', zero_division=0)
        acc_16         = accuracy_score(y_true_16, y_pred_16)

        row_16 = {
            'Model':     name,
            '16T Acc':   round(acc_16, 4),
            '16T F1-M':  round(f1_16_macro, 4),
            '16T F1-W':  round(f1_16_weighted, 4),
        }

        # 16-type AUC-ROC OvR (from 4-axis probabilities)
        if has_probs:
            y_prob_axes = np.column_stack([
                df_r[f'y_prob_{col}'].values for col in LABEL_COLS
            ])
            y_prob_16 = np.zeros((len(df_r), 16), dtype=float)
            DIM_KEYS = ['ie', 'sn', 'tf', 'jp']
            for i in range(len(df_r)):
                for t_idx, t_name in enumerate(MBTI_TYPES):
                    t_upper = t_name.upper()
                    p = 1.0
                    for j, (key, letter) in enumerate(zip(DIM_KEYS, t_upper)):
                        target = DIM_LABEL_MAP[key].get(letter, 0)
                        p *= y_prob_axes[i, j] if target == 1 else (1.0 - y_prob_axes[i, j])
                    y_prob_16[i, t_idx] = p
                row_sum = y_prob_16[i].sum()
                if row_sum > 0:
                    y_prob_16[i] /= row_sum

            try:
                y_true_16_bin = label_binarize(y_true_16, classes=list(range(16)))
                auc_16 = roc_auc_score(y_true_16_bin, y_prob_16,
                                       average='macro', multi_class='ovr')
                row_16['16T AUC'] = round(auc_16, 4)
            except Exception:
                row_16['16T AUC'] = 'N/A'

        all_rows_16.append(row_16)

    df_axis = pd.DataFrame(all_rows_axis).set_index('Model')
    df_16   = pd.DataFrame(all_rows_16).set_index('Model')
    return df_axis, df_16


# ═══════════════════════════════════════════════════════════════
#  DISPLAY RESULTS
# ═══════════════════════════════════════════════════════════════
df_axis, df_16 = build_full_comparison()

print('=' * 90)
print('📊  4-AXIS METRICS — Macro F1 / Accuracy / AUC-ROC on Test Set')
print('=' * 90)
print(df_axis.to_string())

print(f'\n{"=" * 90}')
print('📊  16-TYPE METRICS — Accuracy / Macro F1 / Weighted F1 / AUC-ROC (OvR)')
print('=' * 90)
print(df_16.to_string())

# ═══════════════ TRAINING TIME SUMMARY ═══════════════
print(f'\n{"=" * 90}')
print('⚡ TRAINING TIME SUMMARY (T4×2 Optimized)')
print('=' * 90)
print(f'  {"Model":<20} {"Time":>10}   Optimizations')
print(f'  {"─"*20} {"─"*10}   {"─"*40}')
print(f'  {"SVM + TF-IDF":<20} {t_svm:>8.1f}s   LinearSVC + CalibratedCV(cv=3)')
print(f'  {"RoBERTa-mean":<20} {t_rob_train:>8.1f}s   torch.compile + DataParallel + FP16')
print(f'  {"D-DGCN":<20} {t_dgcn_train:>8.1f}s   DataParallel + FP16 + non_blocking')
print(f'  {"─"*20} {"─"*10}')
print(f'  {"TOTAL":<20} {t_svm + t_rob_train + t_dgcn_train:>8.1f}s')
print(f'\n  Config: BATCH_SIZE={CONFIG["BATCH_SIZE"]} | EVAL_BATCH_SIZE={CONFIG["EVAL_BATCH_SIZE"]}')
print(f'  CUDA:   cudnn.benchmark=True | FP16 mixed precision')
print(f'  Data:   Pre-tokenized | num_workers=4 | persistent_workers')

print(f'\n{"=" * 90}')
print(f'✅ All prediction CSVs saved in: {CONFIG["RESULT_DIR"]}')
print('Files:', os.listdir(CONFIG['RESULT_DIR']))